# Teamfight: isolated TokenRig inference

Select **Runtime → Change runtime type → GPU**. Run cells interactively in order.
This notebook creates a new raw rig, not a qualified combat character. Retargeting
and outfit fitting remain separate local stages. Download the result before the
session ends. No tunnels, daemon, API credentials, Drive access, or deployment.

The T4 compatibility profile was measured on Stafford; this notebook's fresh
bootstrap recipe has not yet been executed end to end. Upstream requires 14 GB
VRAM for its native profile. Hardware availability and runtime limits vary.


In [ ]:
from pathlib import Path
import hashlib, json, os, signal, subprocess, sys, time, uuid, zipfile
BASE = Path('/content') / ('teamfight-tokenrig-' + uuid.uuid4().hex)
BASE.mkdir(exist_ok=False)
TOOLS = BASE / 'tools'
TOOLS.mkdir()

SCRIPTS = {'model-pins.json': '{\n  "repository": "https://github.com/VAST-AI-Research/SkinTokens.git",\n  "commit": "273b691d35989d71cd17ff2895fdc735097b92d1",\n  "license": "MIT",\n  "huggingface": {\n    "repository": "VAST-AI/SkinTokens",\n    "revision": "79736cad0fd84de384d5eede659b4ebd24effe33",\n    "files": [\n      {\n        "path": "experiments/articulation_xl_quantization_256_token_4/grpo_1400.ckpt",\n        "bytes": 1131603979,\n        "sha256": "f4e4706a11cfb520cdde65156a0358545e4fbf8f36237aca01ea5e79d5cb5692"\n      },\n      {\n        "path": "experiments/skin_vae_2_10_32768/last.ckpt",\n        "bytes": 487311745,\n        "sha256": "4843f49e58afff88345806b94ca82e6cc9d8def6e7432e2853c677b154de0ed4"\n      }\n    ]\n  },\n  "qwen_config": {\n    "repository": "Qwen/Qwen3-0.6B",\n    "revision": "c1899de289a04d12100db370d81485cdf75e47ca"\n  },\n  "upstream_sha256": {\n    "src/model/tokenrig.py": "bf27b72201c5fb23028a9f8a8e63b568462abfa2b4e4b15e50529cdbfc15a222",\n    "src/model/skin_vae_model.py": "0d7e3d031bfbad8b4f61493424a7c4538175f30646c7db76564c6412586f3001",\n    "src/model/skin_vae/autoencoders/miche_transformer_blocks.py": "ca2c2ac8885914910e2e7c3297a6f8f31629de41969e552db542325d6a4853bf",\n    "src/server/bpy_server.py": "41071f3e355bab015c0b5a608b52eebc9e223c7e8dc484692a1d0d917c525dd4"\n  }\n}\n', 'download_models.py': '"""Fetch only the pinned inference checkpoints and Qwen configuration."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nfrom huggingface_hub import hf_hub_download, snapshot_download\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'checkout\', type=Path)\n    args = parser.parse_args()\n    pins = json.loads(Path(__file__).with_name(\'model-pins.json\').read_text())\n    for entry in pins[\'huggingface\'][\'files\']:\n        path = Path(hf_hub_download(\n            repo_id=pins[\'huggingface\'][\'repository\'],\n            revision=pins[\'huggingface\'][\'revision\'], filename=entry[\'path\'],\n            local_dir=args.checkout,\n        ))\n        digest = hashlib.sha256()\n        with path.open(\'rb\') as stream:\n            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n                digest.update(chunk)\n        if path.stat().st_size != entry[\'bytes\'] or digest.hexdigest() != entry[\'sha256\']:\n            raise ValueError(f"Pinned checkpoint verification failed: {entry[\'path\']}")\n        print(\'VERIFIED\', entry[\'path\'], flush=True)\n    snapshot_download(\n        repo_id=pins[\'qwen_config\'][\'repository\'], revision=pins[\'qwen_config\'][\'revision\'],\n        local_dir=args.checkout / \'models/Qwen3-0.6B\',\n        allow_patterns=[\'*.json\', \'*.txt\'],\n    )\n\n\nif __name__ == \'__main__\':\n    main()\n', 'prepare_t4_trial.py': '"""Prepare a separate, explicitly experimental FP32/SDPA TokenRig checkout.\n\nNever run this against a UniRig or TRELLIS installation. This is a candidate\nprofile, not a claim of upstream T4 support or a qualified production engine.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\n\nATTENTION = \'\'\'"""FP32 attention for the experimental T4 TokenRig profile."""\nimport torch\nimport torch.nn.functional as F\n\ndef flash_attn_func(q, k, v):\n    # Upstream\'s wrapper expects [B, L, H, D] and a (result, auxiliary) pair.\n    q, k, v = (x.transpose(1, 2) for x in (q, k, v))\n    if q.shape[1] != k.shape[1]:\n        if q.shape[1] % k.shape[1]:\n            raise ValueError("Invalid grouped-query head ratio")\n        repeat = q.shape[1] // k.shape[1]\n        k, v = (x.repeat_interleave(repeat, dim=1) for x in (k, v))\n    # Chunk queries only: each chunk still attends to every key. This bounds\n    # the math fallback\'s temporary allocation without changing its attention.\n    out = torch.cat([\n        F.scaled_dot_product_attention(part, k, v, dropout_p=0.0)\n        for part in q.split(128, dim=2)\n    ], dim=2)\n    return out.transpose(1, 2).contiguous(), None\n\'\'\'\n\n\ndef replace_once(source, before, after):\n    if source.count(before) != 1:\n        raise ValueError(f"Expected one pinned source fragment: {before[:70]}")\n    return source.replace(before, after)\n\n\ndef prepare(root, pins):\n    changes = {}\n    for name, expected in pins[\'upstream_sha256\'].items():\n        raw = (root / name).read_bytes()\n        if hashlib.sha256(raw).hexdigest() != expected:\n            raise ValueError(f"Unreviewed or already patched source: {name}")\n        source = raw.decode(\'utf-8\')\n        if name in (\'src/model/tokenrig.py\', \'src/model/skin_vae_model.py\'):\n            begin = source.index(\'try:\\n    from flash_attn_interface\')\n            end = source.index(\'\\nclass \', begin)\n            source = source[:begin] + \'from src.teamfight_attention import flash_attn_func\\n\' + source[end:]\n            source = source.replace(\'torch.bfloat16\', \'torch.float32\')\n            # FP32 requires no autocast and avoids FP16 overflow of BF16 weights.\n            source = source.replace("@torch.autocast(device_type=\'cuda\', dtype=torch.float32)", \'@torch.autocast(device_type="cuda", enabled=False)\')\n            source = source.replace("@torch.autocast(\'cuda\', dtype=torch.float32)", \'@torch.autocast(device_type="cuda", enabled=False)\')\n            source = source.replace(\'attn_implementation="flash_attention_2"\', \'attn_implementation="sdpa"\')\n        elif name.endswith(\'miche_transformer_blocks.py\'):\n            begin = source.index(\'def flash_attention(q, k, v):\')\n            end = source.index(\'\\nclass MultiheadAttention\', begin)\n            source = source[:begin] + \'\'\'def flash_attention(q, k, v):\n    from src.teamfight_attention import flash_attn_func as sdpa\n    return sdpa(q, k, v)[0]\n\n\'\'\' + source[end:]\n        elif name.endswith(\'bpy_server.py\'):\n            source = replace_once(source, "host=\'0.0.0.0\'", "host=\'127.0.0.1\'")\n        compile(source, name, \'exec\')\n        changes[name] = source\n    changes[\'src/teamfight_attention.py\'] = ATTENTION\n    # Validate every source hash and patch before writing any file.\n    for name, source in changes.items():\n        (root / name).write_text(source, encoding=\'utf-8\', newline=\'\\n\')\n    report = {\n        \'profile\': \'tokenrig-t4-fp32-sdpa-trial-v1\',\n        \'qualified\': False,\n        \'source_commit\': pins[\'commit\'],\n        \'patch_sha256\': hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n        \'files\': {name: hashlib.sha256((root / name).read_bytes()).hexdigest() for name in changes},\n    }\n    (root / \'teamfight-trial-engine.json\').write_text(json.dumps(report, indent=2))\n    return report\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'checkout\', type=Path)\n    args = parser.parse_args()\n    pins = json.loads(Path(__file__).with_name(\'model-pins.json\').read_text())\n    print(json.dumps(prepare(args.checkout.resolve(), pins), indent=2))\n', 'verify_attention.py': '"""Compare the T4 adapter with explicit, noncausal attention on real CUDA."""\nimport argparse\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport torch\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'checkout\', type=Path)\n    args = parser.parse_args()\n    spec = importlib.util.spec_from_file_location(\'trial_attention\', args.checkout / \'src/teamfight_attention.py\')\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    torch.manual_seed(25001)\n    results = []\n    with torch.inference_mode():\n        for heads, kv_heads in [(4, 4), (4, 2)]:\n            # 273 crosses two query boundaries; unequal Q/K lengths catch layout errors.\n            q = torch.randn(2, 273, heads, 16, device=\'cuda\')\n            k, v = (torch.randn(2, 71, kv_heads, 16, device=\'cuda\') for _ in range(2))\n            actual, aux = module.flash_attn_func(q, k, v)\n            repeated_k, repeated_v = (x.repeat_interleave(heads // kv_heads, dim=2) for x in (k, v))\n            scores = torch.einsum(\'bqhd,bkhd->bhqk\', q, repeated_k) / 4\n            expected = torch.einsum(\'bhqk,bkhd->bqhd\', scores.softmax(dim=-1), repeated_v)\n            assert aux is None and actual.shape == q.shape and torch.isfinite(actual).all()\n            torch.testing.assert_close(actual, expected, rtol=2e-5, atol=2e-6)\n            results.append({\'heads\': heads, \'kv_heads\': kv_heads, \'max_error\': (actual - expected).abs().max().item()})\n    print(json.dumps({\'gpu\': torch.cuda.get_device_name(), \'attention_checks\': results}, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'run_trial.py': '"""Run one isolated TokenRig candidate; invoke inside a bounded supervisor."""\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with path.open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'checkout\', type=Path)\n    parser.add_argument(\'source\', type=Path)\n    parser.add_argument(\'output\', type=Path)\n    parser.add_argument(\'--source-sha256\', required=True)\n    parser.add_argument(\'--seed\', type=int, default=25001)\n    parser.add_argument(\'--profile\', choices=[\'native\', \'t4-fp32\'], default=\'native\')\n    parser.add_argument(\'--num-beams\', type=int, default=10)\n    args = parser.parse_args()\n    checkout, source, output = (p.resolve() for p in (args.checkout, args.source, args.output))\n    if sha256(source) != args.source_sha256:\n        raise ValueError(\'Source identity mismatch\')\n    pins = json.loads(Path(__file__).with_name(\'model-pins.json\').read_text())\n    for file in pins[\'huggingface\'][\'files\']:\n        path = checkout / file[\'path\']\n        if path.stat().st_size != file[\'bytes\'] or sha256(path) != file[\'sha256\']:\n            raise ValueError(f"Checkpoint identity mismatch: {file[\'path\']}")\n    if not 1 <= args.num_beams <= 10:\n        raise ValueError(\'Trial beam count must be between 1 and 10\')\n    if args.profile == \'t4-fp32\':\n        engine = json.loads((checkout / \'teamfight-trial-engine.json\').read_text())\n        expected_files = engine[\'files\']\n    else:\n        expected_files = pins[\'upstream_sha256\']\n        engine = {\'profile\': \'tokenrig-native-bf16-fa2-trial-v1\', \'source_commit\': pins[\'commit\'], \'qualified\': False}\n    for name, expected in expected_files.items():\n        if sha256(checkout / name) != expected:\n            raise ValueError(f\'Patched source identity mismatch: {name}\')\n    engine[\'files\'] = {str(p.relative_to(checkout)): sha256(p) for p in sorted((checkout / \'src\').rglob(\'*.py\'))}\n    os.chdir(checkout)\n    sys.path.insert(0, str(checkout))\n    os.environ.update(OMP_NUM_THREADS=\'2\', OPENBLAS_NUM_THREADS=\'2\', GRADIO_ANALYTICS_ENABLED=\'False\', WANDB_MODE=\'disabled\')\n    import random\n    import numpy as np\n    import torch\n    import demo\n    if args.profile == \'native\' and (torch.cuda.get_device_capability()[0] < 8 or not torch.cuda.is_bf16_supported()):\n        raise RuntimeError(\'Native profile requires Ampere or newer and native BF16\')\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    torch.cuda.reset_peak_memory_stats()\n    started = time.monotonic()\n    report = {\n        \'provider\': \'tokenrig\', \'qualified\': False,\n        \'source_sha256\': args.source_sha256, \'seed\': args.seed,\n        \'engine\': engine, \'model_revision\': pins[\'huggingface\'][\'revision\'],\n        \'gpu\': torch.cuda.get_device_name(), \'torch\': torch.__version__,\n        \'precision\': \'float32\' if args.profile == \'t4-fp32\' else \'bfloat16\',\n        \'attention\': \'sdpa\' if args.profile == \'t4-fp32\' else \'flash_attention_2\',\n        \'num_beams\': args.num_beams,\n    }\n    output.parent.mkdir(parents=True, exist_ok=True)\n    # Share our process group so the supervisor\'s deadline also bounds Blender.\n    server = subprocess.Popen([sys.executable, \'bpy_server.py\'])\n    try:\n        demo.wait_for_bpy_server(timeout=90)\n        with torch.inference_mode():\n            demo.load_model(demo.MODEL_CKPTS[0], None)\n            demo.model.eval()\n            demo.run_rig(\n                [source], 5, 0.95, 1.0, 2.0, args.num_beams, False, True, False,\n                [output], demo.MODEL_CKPTS[0], None,\n            )\n        if not output.is_file() or output.stat().st_size < 1000:\n            raise RuntimeError(\'TokenRig did not export a GLB\')\n        report.update(status=\'rigged_unqualified\', artifact_sha256=sha256(output), artifact_bytes=output.stat().st_size)\n    except Exception as error:\n        report.update(status=\'failed\', error=f\'{type(error).__name__}: {error}\')\n        raise\n    finally:\n        server.terminate()\n        try:\n            server.wait(timeout=10)\n        except subprocess.TimeoutExpired:\n            server.kill()\n            server.wait()\n        report.update(seconds=time.monotonic() - started, peak_gpu_mib=torch.cuda.max_memory_allocated() / 1024**2)\n        output.with_suffix(\'.report.json\').write_text(json.dumps(report, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n'}
for name, source in SCRIPTS.items():
    (TOOLS / name).write_text(source)
pins = json.loads(SCRIPTS['model-pins.json'])
print('Isolated workspace:', BASE)
print('Pinned upstream:', pins['commit'])

Install a separate Python 3.11 environment; the notebook kernel is unchanged. Downloads require internet access.

In [ ]:
def run(args, timeout=1800):
    subprocess.run([str(a) for a in args], check=True, timeout=timeout)
run([sys.executable, '-m', 'pip', 'install', 'uv==0.9.5'])
run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', BASE / 'venv'])
PY = BASE / 'venv/bin/python'
def install(*args):
    run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PY, *args])
install('torch==2.7.0', 'torchvision==0.22.0', 'torchaudio==2.7.0',
        '--index-url', 'https://download.pytorch.org/whl/cu128')
CHECKOUT = BASE / 'SkinTokens'
run(['git', 'clone', pins['repository'], CHECKOUT])
run(['git', '-C', CHECKOUT, 'checkout', '--detach', pins['commit']])
install('-r', str(CHECKOUT / 'requirements.txt'), 'bpy==4.5.3',
        'transformers==4.57.6', 'diffusers==0.35.1', 'lightning==2.6.1', 'numpy<3')
hardware = json.loads(subprocess.check_output([str(PY), '-c',
    "import torch,json; assert torch.cuda.is_available(),'GPU runtime required'; print(json.dumps(dict(name=torch.cuda.get_device_name(),free_mib=torch.cuda.mem_get_info()[0]//1048576,major=torch.cuda.get_device_capability()[0],bf16=torch.cuda.is_bf16_supported())))"], text=True))
PROFILE = 'native' if hardware['major'] >= 8 and hardware['bf16'] else 't4-fp32'
minimum = 14*1024 if PROFILE == 'native' else 6*1024
assert hardware['free_mib'] >= minimum, 'Insufficient free VRAM; release other GPU work first'
print(hardware, PROFILE)
if PROFILE == 'native':
    install('packaging', 'ninja', 'wheel', 'setuptools', 'psutil')
    install('flash-attn', '--no-build-isolation')
else:
    run([PY, TOOLS / 'prepare_t4_trial.py', CHECKOUT])
    run([PY, TOOLS / 'verify_attention.py', CHECKOUT])
run([PY, TOOLS / 'download_models.py', CHECKOUT])

Upload one clean A-pose base GLB. Preserve species anatomy. Clothing, hair and weapons belong to later stages.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one base GLB'
name, data = next(iter(uploaded.items()))
assert name.lower().endswith('.glb') and data[:4] == b'glTF', 'A binary glTF mesh is required'
assert len(data) <= 32*1024**2, 'Input exceeds 32 MiB'
SOURCE = BASE / 'source.glb'
SOURCE.write_bytes(data)
SOURCE_SHA = hashlib.sha256(data).hexdigest()
print('Frozen source SHA-256:', SOURCE_SHA)

Run one bounded inference. The supervisor also terminates the local Blender helper on timeout. Output remains unqualified.

In [ ]:
OUTPUT = BASE / 'raw-rig.glb'
LOG = BASE / 'inference.log'
command = [str(PY), str(TOOLS / 'run_trial.py'), str(CHECKOUT), str(SOURCE), str(OUTPUT),
           '--source-sha256', SOURCE_SHA, '--profile', PROFILE]
with LOG.open('w') as log:
    process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    try:
        result = process.wait(timeout=900)
    finally:
        try: os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError: pass
        time.sleep(1)
        try: os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError: pass
print(LOG.read_text()[-5000:])
assert result == 0 and OUTPUT.exists(), 'Inference failed; retain the log'
print((BASE / 'raw-rig.report.json').read_text())

In [ ]:
freeze = subprocess.check_output([str(sys.executable), '-m', 'uv', 'pip', 'freeze', '--python', str(PY)], text=True)
(BASE / 'environment.txt').write_text(freeze)
(BASE / 'manifest.json').write_text(json.dumps({
    'provider': 'tokenrig', 'qualified': False, 'source_sha256': SOURCE_SHA,
    'upstream_commit': pins['commit'], 'profile': PROFILE, 'hardware': hardware,
    'tools': {name: hashlib.sha256(source.encode()).hexdigest() for name, source in SCRIPTS.items()},
    'raw_sha256': hashlib.sha256(OUTPUT.read_bytes()).hexdigest()}, indent=2))
archive = BASE / 'tokenrig-result.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for name in ['raw-rig.glb', 'raw-rig.report.json', 'manifest.json', 'environment.txt', 'inference.log']:
        bundle.write(BASE / name, name)
files.download(str(archive))